# 第13周：完整框架搭建、UI（开始/结束界面）（Day 85-91）

这是一个可以自动检测练习题的 Jupyter Notebook，配合《第13周详细课程》使用。

**这个notebook每一天都包含两部分：📖 知识点讲解（复习本周课程的讲解要点和实际场景）+ ✏️ 练习题与🔍自动检测。**

**怎么用这个notebook：**
1. 从上到下按顺序运行每一个cell（选中cell后按 `Shift + Enter`）
2. 看到 `✏️ 你的代码` 的cell，把 `pass` 删掉，写上你的代码
3. 写完之后，运行下面紧跟着的 `🔍 检测` cell，看看是不是显示 ✅（通过）
4. 如果显示 ❌，看看提示信息，回去改一改代码再重新运行这两个cell（先改代码的cell，再改检测的cell）

**一个小提醒：** 这周开始正式进入"期末大项目"阶段了！这个notebook里涉及的"鼠标点击"没办法用真实鼠标来自动测试，所以继续用"构造一个假设的点击事件（指定点击发生在哪个坐标）"的方式来测试——核心知识点（`collidepoint`、`MOUSEBUTTONDOWN`事件、按钮绘制）跟真实写法完全一样。这周的Checkpoint会把第10-13周的所有内容串成一个真正能从头玩到尾、还能重新开始的游戏框架，非常建议完成后在自己电脑上用真实的窗口和鼠标点一点、玩一玩！

这个notebook不会给你答案，只会告诉你做得对不对，加油！


## 📑 本周内容导航

每天的内容都是先有 **📖 知识点讲解**，再有 **✏️ 练习 + 🔍 检测**，请按顺序学习：

- Day 85：完整主循环结构——根据状态决定这一帧该做什么
- Day 86：开始界面——标题与按钮
- Day 87：鼠标点击检测
- Day 88：结束界面——显示分数与重新开始按钮
- Day 89：多界面切换整合
- Day 90：重新开始功能——重置游戏状态
- Day 91：第13周 Checkpoint


In [ ]:
# 🔧 工具cell：检测练习题用的工具函数和pygame初始化，不需要修改，直接运行一次就好
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # 让pygame可以在没有真实显示器的环境里运行（这个notebook专用，自己电脑上写游戏不需要这一行）
os.environ["SDL_AUDIODRIVER"] = "dummy"
import pygame
import random
pygame.init()
pygame.font.init()

def test_check(fn, desc):
    """fn 是一个不带参数的函数（通常是 lambda），运行它并根据结果打印 ✅ 或 ❌"""
    try:
        result = fn()
    except Exception as e:
        print(f"❌ {desc} —— 运行出错：{type(e).__name__}: {e}")
        return False
    ok = bool(result)
    if ok:
        print(f"✅ {desc}")
    else:
        print(f"❌ {desc}")
    return ok

print("工具函数准备好了，可以开始做练习题了！")


## Day 85：完整主循环结构——根据状态决定这一帧该做什么

**学习目标**：理解一个完整的游戏需要哪几个界面，掌握怎么根据当前状态决定该画哪个画面

### 📖 知识点讲解

**讲解要点**

- **一个真正完整的游戏，不是只有"玩的那部分"，还需要"开始界面"和"结束界面"**。前面做的"游戏核心玩法"，其实只是整个游戏里`playing`这一个状态；这周要把`menu`和`game_over`这两个状态也做成真正"看得见、能点击"的画面。
  **实际场景**：手机App很少是"打开就直接进入核心功能"的，几乎都会先有一个启动页，完成某个流程后也会有一个"完成"页面——"核心功能"只是整个使用流程中的一段。

- **因为游戏有好几种状态，"每一帧该画什么、该处理什么输入"也应该跟着状态变化**：`menu`状态该画标题和按钮；`playing`状态该画游戏画面；`game_over`状态该画最终分数和"重新开始"按钮。可以用一个"状态 → 该做的事"的对照表（字典）来管理这种对应关系。
  **实际场景**：自动售货机在"等待选择"状态下接受"按商品按钮"，在"吐货中"状态下这个按钮暂时没反应——机器会根据"自己现在是什么状态"，决定现在该响应哪种操作。


### ✏️ 练习1：`get_screen_for_state(state)`

接收游戏状态`state`，用一个字典把状态映射成对应的画面名字：`"menu"`对应`"start_screen"`，`"playing"`对应`"game_screen"`，`"game_over"`对应`"end_screen"`；如果传入的状态不在这几种里面，`return "unknown"`。（提示：字典的`.get(key, 默认值)`方法可以很方便地处理"找不到就返回默认值"这种情况。）


In [ ]:
# ✏️ 你的代码
def get_screen_for_state(state):
    pass


In [ ]:
# 🔍 检测
results_85_1 = []
results_85_1.append(test_check(lambda: get_screen_for_state("menu") == "start_screen", "menu状态对应start_screen"))
results_85_1.append(test_check(lambda: get_screen_for_state("game_over") == "end_screen", "game_over状态对应end_screen"))
results_85_1.append(test_check(lambda: get_screen_for_state("???") == "unknown", "不认识的状态返回unknown"))
print(f"\n通过 {sum(results_85_1)}/{len(results_85_1)} 项")


### ✏️ 挑战：`draw_screen_by_state(surface, state, bg_colors)`

接收画布`surface`、当前状态`state`、一个"状态→背景色"的字典`bg_colors`。从`bg_colors`里找出当前状态对应的颜色（找不到就用`(128, 128, 128)`灰色作为默认值），用这个颜色填充`surface`，`return surface`。


In [ ]:
# ✏️ 你的代码
def draw_screen_by_state(surface, state, bg_colors):
    pass


In [ ]:
# 🔍 检测
results_85_2 = []
_surf85 = pygame.Surface((50, 50))
draw_screen_by_state(_surf85, "menu", {"menu": (10, 20, 30), "playing": (0, 0, 0)})
results_85_2.append(test_check(lambda: _surf85.get_at((5, 5)) == (10, 20, 30), "根据状态填充了正确的背景色"))
print(f"\n通过 {sum(results_85_2)}/{len(results_85_2)} 项")


## Day 86：开始界面——标题与按钮

**学习目标**：掌握怎么用矩形+文字做出一个按钮的视觉效果，并计算按钮在屏幕上居中的位置

### 📖 知识点讲解

**讲解要点**

- **游戏里的"按钮"，在最简单的版本里，其实就是一个画了背景色的矩形，中间叠加了一些文字**——用前面学过的`pygame.draw.rect`和`pygame.font`组合起来就能做出一个能用的按钮。
  **实际场景**：很多小程序、小工具的界面，本质上也是用一个个"矩形区域+文字/图标"拼出来的，只是设计得更精美——核心结构是一样的。

- **文字一般需要"居中"地贴在按钮矩形里**，`文字图像.get_rect(center=按钮矩形.center)`可以直接算出"让文字图像的中心对齐按钮矩形的中心"所需要的位置。
  **实际场景**：海报设计时，标题文字通常会"居中"对齐在某个区域正中间——设计软件里的"居中对齐"工具按钮，做的就是这种计算。

- **按钮在屏幕上的位置，通常希望"水平居中"**：`按钮的x坐标 = (屏幕宽度 - 按钮宽度) ÷ 2`——这跟算一段文字在一行里居中对齐是同一个数学道理。
  **实际场景**：网页上的"登录"按钮、弹窗里的"确定"按钮，几乎都是水平居中摆放的，背后就是这个简单的居中公式。


### ✏️ 练习1：`draw_button(surface, rect, color, label, font)`

接收画布`surface`、按钮矩形`rect`、背景颜色`color`、按钮上的文字`label`、字体对象`font`。先用`pygame.draw.rect`把`rect`这块区域画成`color`颜色，再用`font.render(label, True, (255, 255, 255))`渲染出文字图像，用`文字图像.get_rect(center=rect.center)`算出居中的位置，`blit`上去，`return surface`。


In [ ]:
# ✏️ 你的代码
def draw_button(surface, rect, color, label, font):
    pass


In [ ]:
# 🔍 检测
results_86_1 = []
try:
    _font86 = pygame.font.SysFont(None, 24)
    _btn_rect86 = pygame.Rect(50, 50, 100, 30)
    _surf86 = pygame.Surface((200, 150)); _surf86.fill((0, 0, 0))
    draw_button(_surf86, _btn_rect86, (0, 150, 0), "开始", _font86)
    results_86_1.append(test_check(lambda: _surf86.get_at((51, 51)) == (0, 150, 0), "按钮的背景颜色正确"))
    results_86_1.append(test_check(lambda: any(_surf86.get_at((x, y))[:3] not in [(0, 0, 0), (0, 150, 0)] for x in range(60, 140) for y in range(55, 75)), "按钮上的文字被画出来了"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_86_1.extend([False, False])
print(f"\n通过 {sum(results_86_1)}/{len(results_86_1)} 项")


### ✏️ 挑战：`make_centered_button_rect(screen_w, screen_h, button_w, button_h, y)`

接收窗口宽高`screen_w`、`screen_h`，按钮宽高`button_w`、`button_h`，以及按钮的`y`坐标。算出能让按钮"水平居中"的`x`坐标，`return pygame.Rect(x, y, button_w, button_h)`。


In [ ]:
# ✏️ 你的代码
def make_centered_button_rect(screen_w, screen_h, button_w, button_h, y):
    pass


In [ ]:
# 🔍 检测
results_86_2 = []
_rect86b = make_centered_button_rect(300, 200, 100, 40, 80)
results_86_2.append(test_check(lambda: _rect86b == pygame.Rect(100, 80, 100, 40), "按钮矩形的位置计算正确(水平居中)"))
print(f"\n通过 {sum(results_86_2)}/{len(results_86_2)} 项")


## Day 87：鼠标点击检测

**学习目标**：掌握怎么判断一个点是否落在某个矩形范围内，掌握检测鼠标点击事件的方法

### 📖 知识点讲解

**讲解要点**

- **`pygame.Rect`自带一个`collidepoint(点)`方法，可以判断某一个点（比如鼠标点击的位置）是否落在这个矩形范围内**，返回`True`或`False`。这跟第11周学的`colliderect`很像，只是这次判断的是"一个点"而不是"另一个矩形"。
  **实际场景**：触屏手机判断你是不是点到了某个图标，本质上就是在判断"你手指点击的坐标"有没有落在"这个图标所占的矩形范围"内。

- **鼠标点击在pygame里也是一种事件，类型是`pygame.MOUSEBUTTONDOWN`**，这个事件对象有一个`.pos`属性，记录了"点击发生时鼠标所在的坐标"。判断"玩家有没有点到某个按钮"，就是在事件列表里找有没有`MOUSEBUTTONDOWN`事件，并且它的`.pos`恰好落在按钮的矩形范围内。
  **实际场景**：跟第10周讲过的`KEYDOWN`事件记录"按了哪个键"类似，`MOUSEBUTTONDOWN`事件记录的是"点击发生在哪个位置"。


### ✏️ 练习1：`is_point_in_rect(point, rect)`

接收一个坐标`point`（比如`(x, y)`元组）和一个矩形`rect`，用`pygame.Rect(rect).collidepoint(point)`判断这个点是否在矩形范围内，`return`结果。


In [ ]:
# ✏️ 你的代码
def is_point_in_rect(point, rect):
    pass


In [ ]:
# 🔍 检测
results_87_1 = []
results_87_1.append(test_check(lambda: is_point_in_rect((55, 55), pygame.Rect(50, 50, 20, 20)) == True, "点在矩形内部时返回True"))
results_87_1.append(test_check(lambda: is_point_in_rect((5, 5), pygame.Rect(50, 50, 20, 20)) == False, "点在矩形外部时返回False"))
print(f"\n通过 {sum(results_87_1)}/{len(results_87_1)} 项")


### ✏️ 挑战：`get_clicked_button(events, button_rect)`

接收事件列表`events`和按钮矩形`button_rect`，用循环检查里面有没有`pygame.MOUSEBUTTONDOWN`类型的事件，而且它的`.pos`落在`button_rect`范围内（可以用Day87练习1的函数）。只要找到一个这样的事件，`return True`；循环结束都没找到就`return False`。


In [ ]:
# ✏️ 你的代码
def get_clicked_button(events, button_rect):
    pass


In [ ]:
# 🔍 检测
results_87_2 = []
_btn_rect87 = pygame.Rect(100, 80, 100, 40)
_events87a = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 90), button=1)]
_events87b = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(5, 5), button=1)]
results_87_2.append(test_check(lambda: get_clicked_button(_events87a, _btn_rect87) == True, "点击在按钮范围内时检测到点击"))
results_87_2.append(test_check(lambda: get_clicked_button(_events87b, _btn_rect87) == False, "点击在按钮范围外时没有检测到点击"))
print(f"\n通过 {sum(results_87_2)}/{len(results_87_2)} 项")


## Day 88：结束界面——显示分数与重新开始按钮

**学习目标**：掌握怎么在结束界面显示最终得分，并复用按钮点击检测来判断重新开始是否被点击

### 📖 知识点讲解

**讲解要点**

- **结束界面最重要的两件事：告诉玩家"最终得分是多少"，以及给玩家一个"重新开始"的入口**。显示分数复用第12周学过的`pygame.font`文字渲染技巧即可，只是这次显示的文字内容变成"游戏结束！最终得分：xxx"这样总结性的信息。
  **实际场景**：很多小游戏结束时会弹出一个总结页面，上面写着"本次得分：xxx，是否要再来一局？"——这几乎是所有"一局一局玩"的游戏共同的收尾设计。

- **"重新开始"按钮的点击检测，跟开始按钮检测用的是完全一样的方法**——一旦掌握了"判断点击有没有落在某个按钮范围内"这个通用技巧，不管这个按钮叫"开始"还是"重新开始"，检测的写法都是可以复用的。
  **实际场景**：手机里很多App的"确定"和"取消"按钮，虽然文字、颜色不一样，但底层"判断有没有点到这个按钮"的检测逻辑写法几乎一模一样。


### ✏️ 练习1：`render_game_over_text(surface, font, score, x, y, color)`

接收画布、字体对象、最终得分`score`、位置`x`、`y`、颜色`color`。用`font.render(f"游戏结束！最终得分: {score}", True, color)`渲染文字，`blit`到`(x, y)`位置，`return surface`。


In [ ]:
# ✏️ 你的代码
def render_game_over_text(surface, font, score, x, y, color):
    pass


In [ ]:
# 🔍 检测
results_88_1 = []
try:
    _surf88 = pygame.Surface((200, 150)); _surf88.fill((0, 0, 0))
    render_game_over_text(_surf88, _font86, 50, 10, 10, (255, 255, 255))
    results_88_1.append(test_check(lambda: any(_surf88.get_at((x, y))[:3] != (0, 0, 0) for x in range(10, 190) for y in range(10, 30)), "结束画面的文字被画出来了"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_88_1.append(False)
print(f"\n通过 {sum(results_88_1)}/{len(results_88_1)} 项")


### ✏️ 挑战：`handle_end_screen_click(events, restart_rect)`

接收事件列表`events`和重新开始按钮的矩形`restart_rect`，用Day87练习2的`get_clicked_button`判断有没有点击这个按钮：点击了就`return "restart"`，没点击就`return "none"`。


In [ ]:
# ✏️ 你的代码
def handle_end_screen_click(events, restart_rect):
    pass


In [ ]:
# 🔍 检测
results_88_2 = []
_restart_rect88 = pygame.Rect(100, 100, 100, 40)
_events88a = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 110), button=1)]
_events88b = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(5, 5), button=1)]
results_88_2.append(test_check(lambda: handle_end_screen_click(_events88a, _restart_rect88) == "restart", "点击重新开始按钮返回restart"))
results_88_2.append(test_check(lambda: handle_end_screen_click(_events88b, _restart_rect88) == "none", "没点到按钮时返回none"))
print(f"\n通过 {sum(results_88_2)}/{len(results_88_2)} 项")


## Day 89：多界面切换整合

**学习目标**：练习把根据状态画不同的画面和根据状态处理不同的点击分别整理成统一的函数

### 📖 知识点讲解

**讲解要点**

- **这一天要做的是把"画面"和"点击处理"分别"收拢"成两个统一的函数**：一个专门负责"根据当前状态，画出对应的画面"，另一个专门负责"根据当前状态和发生的点击，判断应该触发什么动作"。这样以后主循环调用起来会非常清晰，不需要到处写一堆`if state == ...`。
  **实际场景**：很多表单软件会把"界面怎么显示"和"用户操作怎么处理"分开设计成两套独立的逻辑——"显示逻辑"只管画出当前该看到的东西，"处理逻辑"只管响应用户的操作。

- **"点击之后应该触发什么动作"，可以直接用一个简单的字符串来表示**（比如`"start"`、`"restart"`、`"none"`），而不是直接在检测函数里就把状态改掉——检测函数只负责"判断发生了什么"，至于"具体要怎么改变游戏状态"，交给调用它的地方来决定。
  **实际场景**：客服系统里，"识别客户想做什么"和"真正去执行操作"通常是两个独立的步骤——先分类判断，再分派处理。


### ✏️ 练习1：`draw_current_screen(surface, state, font, score, button_rect, colors)`

接收画布、当前状态`state`、字体、分数`score`、按钮矩形`button_rect`、"状态→颜色"字典`colors`。如果`state == "menu"`：用`colors["menu"]`填充背景，再用Day86的`draw_button`画一个颜色`(0, 150, 0)`、文字`"开始游戏"`的按钮；如果`state == "playing"`：只用`colors["playing"]`填充背景；如果`state == "game_over"`：用`colors["game_over"]`填充背景，再用Day88的`render_game_over_text`画出分数文字（位置用`(10, 10)`，颜色用`(255, 255, 255)`）。最后`return surface`。


In [ ]:
# ✏️ 你的代码
def draw_current_screen(surface, state, font, score, button_rect, colors):
    pass


In [ ]:
# 🔍 检测
results_89_1 = []
try:
    _start_rect89 = pygame.Rect(100, 80, 100, 40)
    _colors89 = {"menu": (10, 10, 10), "playing": (0, 0, 0), "game_over": (60, 0, 0)}
    _surf89a = pygame.Surface((300, 200))
    draw_current_screen(_surf89a, "menu", _font86, 0, _start_rect89, _colors89)
    results_89_1.append(test_check(lambda: _surf89a.get_at((1, 1)) == (10, 10, 10), "menu状态画出了正确的背景"))
    results_89_1.append(test_check(lambda: _surf89a.get_at((101, 81)) == (0, 150, 0), "menu状态画出了开始按钮"))
    _surf89b = pygame.Surface((300, 200))
    draw_current_screen(_surf89b, "game_over", _font86, 30, _start_rect89, _colors89)
    results_89_1.append(test_check(lambda: _surf89b.get_at((1, 1)) == (60, 0, 0), "game_over状态画出了正确的背景"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_89_1.extend([False, False, False])
print(f"\n通过 {sum(results_89_1)}/{len(results_89_1)} 项")


### ✏️ 挑战：`handle_click_for_state(state, events, start_button_rect, restart_button_rect)`

接收当前状态`state`、事件列表`events`、开始按钮矩形、重新开始按钮矩形。如果`state == "menu"`并且点击到了开始按钮，`return "start"`；如果`state == "game_over"`并且点击到了重新开始按钮，`return "restart"`；其他情况都`return "none"`。


In [ ]:
# ✏️ 你的代码
def handle_click_for_state(state, events, start_button_rect, restart_button_rect):
    pass


In [ ]:
# 🔍 检测
results_89_2 = []
_restart_rect89 = pygame.Rect(100, 100, 100, 40)
results_89_2.append(test_check(lambda: handle_click_for_state("menu", _events87a, _start_rect89, _restart_rect89) == "start", "menu状态点击开始按钮返回start"))
results_89_2.append(test_check(lambda: handle_click_for_state("game_over", _events88a, _start_rect89, _restart_rect89) == "restart", "game_over状态点击重新开始按钮返回restart"))
results_89_2.append(test_check(lambda: handle_click_for_state("playing", _events87a, _start_rect89, _restart_rect89) == "none", "playing状态下点击不会触发任何动作"))
print(f"\n通过 {sum(results_89_2)}/{len(results_89_2)} 项")


## Day 90：重新开始功能——重置游戏状态

**学习目标**：掌握怎么把一局游戏的所有数据归零，恢复到可以重新开始的状态

### 📖 知识点讲解

**讲解要点**

- **"重新开始"按钮被点击之后，需要把上一局游戏留下的所有痕迹都清理掉**：分数归零、miss次数归零、等级和速度恢复到最初的样子，场上还没被处理的掉落物也要清空，状态要切回`playing`。如果漏掉了某一项没有重置，下一局游戏开局就会带着"上一局的尾巴"。
  **实际场景**：游戏机"重新开始"的时候，存档里"这一局"相关的临时数据也都会被清零——如果忘了清理某一项数据，玩家会发现"明明是新的一局，怎么数字还是上一局的"，这是真实游戏开发里很常见的一类bug。

- **判断"要不要执行重置"，需要同时满足两个条件：当前确实在`game_over`状态，而且确实检测到了点击"重新开始"按钮这个事件**——把"状态对不对"和"点击检测对不对"两个条件都检查到，才是稳妥的写法。
  **实际场景**：自动门只有在"检测到有人靠近"**并且**"门确实是关闭状态"的时候才会打开——需要同时满足所有必要的条件才执行对应的操作。


### ✏️ 练习1：`reset_game_stats(game)`

接收一个游戏对象`game`（它已经有`score`、`misses`、`level`、`speed`、`base_speed`、`state`、`items`这些属性），把`game.score`、`game.misses`重置为`0`，`game.level`重置为`1`，`game.speed`重置为`game.base_speed`，用`game.items.empty()`清空掉落物精灵组，把`game.state`设成`"playing"`，`return game`。


In [ ]:
# ✏️ 你的代码
def reset_game_stats(game):
    pass


In [ ]:
# 🔍 检测
results_90_1 = []
class _FakeGame90:
    pass
try:
    _g90 = _FakeGame90()
    _g90.score = 50; _g90.misses = 2; _g90.level = 3; _g90.speed = 40; _g90.base_speed = 20; _g90.state = "game_over"
    _g90.items = pygame.sprite.Group()
    reset_game_stats(_g90)
    results_90_1.append(test_check(lambda: (_g90.score, _g90.misses, _g90.level, _g90.speed, _g90.state) == (0, 0, 1, 20, "playing"), "重置后各项数据都恢复到初始值"))
    results_90_1.append(test_check(lambda: len(_g90.items) == 0, "重置后掉落物也被清空了"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_90_1.extend([False, False])
print(f"\n通过 {sum(results_90_1)}/{len(results_90_1)} 项")


### ✏️ 挑战：`process_restart_click(game, events, restart_button_rect)`

接收游戏对象`game`、事件列表`events`、重新开始按钮矩形。如果`game.state == "game_over"`**并且**点击到了重新开始按钮，就调用Day90练习1的`reset_game_stats`重置游戏，`return True`；否则什么都不做，`return False`。


In [ ]:
# ✏️ 你的代码
def process_restart_click(game, events, restart_button_rect):
    pass


In [ ]:
# 🔍 检测
results_90_2 = []
try:
    _g90b = _FakeGame90()
    _g90b.score = 50; _g90b.misses = 2; _g90b.level = 3; _g90b.speed = 40; _g90b.base_speed = 20; _g90b.state = "game_over"
    _g90b.items = pygame.sprite.Group()
    _restart_rect90 = pygame.Rect(100, 100, 100, 40)
    _events90 = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 110), button=1)]
    results_90_2.append(test_check(lambda: process_restart_click(_g90b, _events90, _restart_rect90) == True, "在game_over状态点击重新开始按钮会触发重置"))
    results_90_2.append(test_check(lambda: _g90b.score == 0 and _g90b.state == "playing", "触发重置后数据确实被重置了"))

    _g90c = _FakeGame90()
    _g90c.score = 50; _g90c.misses = 2; _g90c.level = 3; _g90c.speed = 40; _g90c.base_speed = 20; _g90c.state = "playing"
    _g90c.items = pygame.sprite.Group()
    results_90_2.append(test_check(lambda: process_restart_click(_g90c, _events90, _restart_rect90) == False, "不在game_over状态时，点击不会触发重置"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_90_2.extend([False, False, False])
print(f"\n通过 {sum(results_90_2)}/{len(results_90_2)} 项")


## Day 91：第13周 Checkpoint

### 小测验（口头回答，答案由家长核对，这里不自动检测）

1. 一个完整的游戏，除了"正在玩"这个状态，通常还需要哪两个界面？分别负责什么？
2. `collidepoint`是用来判断什么的？跟第11周学的`colliderect`有什么区别？
3. 为什么"判断点击了什么"和"点击之后具体要做什么"最好分成两步，而不是在检测的时候就直接改状态？
4. "重新开始"的时候，哪些数据需要被重置？如果漏掉了某一项会发生什么？
5. 判断"是否应该执行重置"，为什么需要同时检查"当前状态"和"有没有点击事件"这两个条件？

### 挑战题：`make_complete_game(screen_w, screen_h, player_w, player_h, item_size, base_speed, points_each, score_per_level, increase_per_level, start_button_rect, restart_button_rect, player_speed=15, max_misses=3)` —— 把开始界面、核心玩法、结束界面连成完整的一局

**背景**：这是期末大项目正式开始后的第一次整合——要把本周做的"开始界面""点击检测""结束界面""重置功能"，跟第10-12周做的"移动、碰撞、计分、难度递增"全部串起来，做出一个从头到尾能完整玩一整局、还能重新开始的游戏框架。

**要求**：在函数内部定义一个表示"完整游戏"的类，至少包含：

1. **三种状态**：`self.state`初始是`"menu"`，保存好传入的`start_button_rect`、`restart_button_rect`
2. **`handle_events(self, events)`方法**：用Day89的`handle_click_for_state`判断这一帧该做什么动作——`"start"`就把`self.state`改成`"playing"`；`"restart"`就调用Day90的`reset_game_stats(self)`重置整个游戏
3. **核心玩法**：复用第10-12周做过的内容——`move_player`（移动+边界检测）、`spawn_item`（生成掉落物）、`step`（**只有在`playing`状态才处理**碰撞检测、计分、miss判断、难度递增这些逐帧更新的内容）
4. **初始数据**：`score=0`、`misses=0`、`level=1`、`speed=base_speed`、`items`是一个空的精灵组、玩家矩形在窗口底部中间

函数最后创建并`return`一个这个类的对象。

**完成标准**
- [ ] 开始界面和结束界面都能通过点击正确触发状态切换（`menu`→`playing`、`game_over`→`playing`）
- [ ] 核心玩法（移动、碰撞、计分、难度递增、miss判断）没有明显bug，表现跟第10-12周学的完全一致
- [ ] 点击"重新开始"之后，所有数据都被正确重置，不会带着上一局的痕迹
- [ ] 整个游戏能完整玩一整局：从`menu`开始，玩到`game_over`，再重新开始，循环顺畅


In [ ]:
# ✏️ 你的代码
def make_complete_game(screen_w, screen_h, player_w, player_h, item_size, base_speed, points_each,
                        score_per_level, increase_per_level, start_button_rect, restart_button_rect,
                        player_speed=15, max_misses=3):
    pass


In [ ]:
# 🔍 检测
results_91 = []

try:
    _start_rect91 = pygame.Rect(100, 80, 100, 40)
    _restart_rect91 = pygame.Rect(100, 100, 100, 40)
    _game91 = make_complete_game(300, 200, 40, 10, 20, 20, 10, 20, 10, _start_rect91, _restart_rect91, player_speed=15, max_misses=3)
    results_91.append(test_check(lambda: _game91.state == "menu", "新游戏从menu状态开始"))
except Exception as e:
    print(f"❌ 初始状态测试运行出错：{type(e).__name__}: {e}")
    results_91.append(False)

try:
    _click_start_events = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 90), button=1)]
    _game91.handle_events(_click_start_events)
    results_91.append(test_check(lambda: _game91.state == "playing", "点击开始按钮后进入playing状态"))
except Exception as e:
    print(f"❌ 开始按钮测试运行出错：{type(e).__name__}: {e}")
    results_91.append(False)

try:
    _game91.spawn_item(x=0)
    _game91.spawn_item(x=280)
    _game91.spawn_item(x=50)
    for _ in range(11):
        _game91.step()
    results_91.append(test_check(lambda: _game91.misses == 3 and _game91.state == "game_over", "miss次数够多之后正确进入game_over状态"))
except Exception as e:
    print(f"❌ game_over测试运行出错：{type(e).__name__}: {e}")
    results_91.append(False)

try:
    _click_restart_events = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 110), button=1)]
    _game91.handle_events(_click_restart_events)
    results_91.append(test_check(lambda: _game91.state == "playing" and _game91.score == 0 and _game91.misses == 0, "点击重新开始按钮后，游戏正确重置并回到playing状态"))
except Exception as e:
    print(f"❌ 重新开始测试运行出错：{type(e).__name__}: {e}")
    results_91.append(False)

try:
    _game91.spawn_item(x=130)
    for _ in range(9):
        _game91.step()
    results_91.append(test_check(lambda: _game91.score == 10, "重新开始之后，接住掉落物依然能正确加分(主要玩法没有被破坏)"))
except Exception as e:
    print(f"❌ 重新开始后玩法测试运行出错：{type(e).__name__}: {e}")
    results_91.append(False)

print()
passed = sum(results_91); total = len(results_91)
print(f"第13周Checkpoint：通过 {passed}/{total} 项")
if passed == total:
    print("🎉 第13周完成！记得让家长核对一下小测验的口头回答。恭喜做出了一个完整可玩的游戏框架，最后一周就要给它搭配真正的画面细节、音效、调试优化，正式完工啦！")


---

恭喜完成第13周的所有练习，也恭喜做出了一个真正"能从头玩到尾"的游戏框架！这周把开始界面、核心玩法、结束界面、重新开始这几个环节完整串了起来，这正是一个"完整游戏"最基本的骨架。如果有检测没通过，可以回去看看对应的"讲解要点"和"实际场景"部分，想一想问题出在哪里，也强烈建议在自己电脑上用真实的窗口和鼠标把这个框架跑起来，从头玩到尾试一试。最后一周就是最终的调试优化和发布啦！
